In [1]:
%run ../../Latex_macros.ipynb

<IPython.core.display.Latex object>

# RLLM-4: Case study: training a reasoning model

## Where this session fits

> **The limitation we start from: RLHF and RLAIF rewards are learned proxies, and a proxy can be exploited. For some tasks a program can check the answer.**

- **The question:** how is a reasoning model trained end to end with verifiable rewards?
- **Position: session 4 of the RL for LLMs series (RLLM-1 to RLLM-13)**
    - RLLM-2 and RLLM-3 built rewards from comparisons
    - this session uses a reward a program checks, with plain GRPO from C2
    - the next sessions go deeper: RLLM-5 on PPO, RLLM-6 on GRPO and its refinements
- **Series goal:** turn a pretrained language model into a useful, aligned assistant with reinforcement learning, and know the methods well enough to choose among them

**By the end you can**

1. say what a reasoning model does differently, and why that behavior is instilled in post-training
2. compute a rule-based reward and its group advantage for a group of completions
3. say why reasoning RL starts from an SFT step, and read the stages of the DeepSeek-R1 recipe
4. say why a rule-based reward was preferred to a learned one there
5. tell an outcome reward from a process reward


# What a reasoning model does

**Why this section?** The case study trains one behavior: thinking before answering.

> **A reasoning model writes a long chain of thought, with reflection and revision, before it gives its answer.**

- the **chain of thought** is step-by-step reasoning in the completion
- **reflection:** looking back at the work so far, and judging the solution and the strategy
- **revision:** changing the current approach
- for practical reasons the completion has two marked parts
    - the reasoning, between `<think>` and `</think>`
    - the answer, between `<answer>` and `</answer>`
    - the markers let a program find the answer, and let the user hide the reasoning
- this is not the natural behavior of a pretrained model: it is **instilled in post-training**

**A running example**

- prompt: "Find a number that, multiplied by 3 and then increased by 7, equals 22."
- a base model's completion: "The answer is 5 because 15 divided by 3 is 5"
- a reasoning completion
    - `<think>` opens the reasoning
        - let the number be x, so 3x + 7 = 22
        - subtract 7: 3x = 15
        - divide by 3: x = 5
    - `</think>` closes it, then `<answer>5</answer>`


## The reward: rules, not a learned model

**Why this section?** This is the point of the case study: the reward is a program.

> **The reward is rule-based: a sum of checks on the completion, computed by a program with nothing to train.**

- an **accuracy** reward: is the final answer correct?
    - a **verifiable** question: we may not know how to find the answer, but given it we can check it
    - for example: a number, a prime factorization, a program that passes its tests
- a **format** reward: are the `<think>` and `<answer>` markers present and in order?
    - without them the answer cannot be found in the completion
- other terms are possible: a penalty for very long completions, a reward for staying in one language
- why not a learned reward model (RLLM-2)?
    - the DeepSeek-R1 paper reports that a neural reward model was susceptible to reward hacking in large-scale RL
    - a program has no parameters, and no need for human or AI labelers


## A worked group: rewards and advantages

**Why this section?** The group advantage of C2, applied to a rule-based reward.

> **Rewards are centered and scaled within the group, so the policy is pushed toward better completions, not just good ones.**

- one prompt, $G = 5$ completions, each with its three reward terms (an illustrative reward)
| completion | accuracy | format | length penalty | total reward | advantage |
|:--|:--|:--|:--|:--|:--|
| 1 | 1.0 | 1.0 | -0.10 | 1.90 | +0.91 |
| 2 | 1.0 | 1.0 | -0.50 | 1.50 | +0.02 |
| 3 | 0.0 | 1.0 | -0.10 | 0.90 | -1.31 |
| 4 | 1.0 | 1.0 | -0.05 | 1.95 | +1.02 |
| 5 | 1.0 | 1.0 | -0.80 | 1.20 | -0.64 |

- the mean of the totals is $1.49$ and the sample standard deviation is $0.45$ (divisor $G - 1$, as in C2)
- the advantage is $(\text{total} - \text{mean}) / \text{std}$, the same for every token of a completion


**Reading the table**

- completion 3 has the wrong answer: its accuracy is $0$, and its advantage is far below the others
- completions 1, 2, 4 and 5 are all correct, but their advantages differ
    - completions 1 and 4 are clearly above the group mean
    - completion 2 sits on the mean: correct, but long
    - completion 5 is correct but longer: its length penalty puts it below the mean
- all rewards here are positive: without centering, every completion would be pushed up
- if every completion earned the same total, every advantage would be $0$, and nothing would be learned (C2)


## Why reasoning RL often starts from SFT: the cold start

**Why this section?** RL raises the probability of what the model already does sometimes (C1). A base model may never do this.

> **If no sampled completion has the right format, every reward is zero and there is nothing to learn from: a short SFT step first.**

- in the running example the base model has almost no chance of writing the markers
    - its training text never contained them
- GRPO needs some completions in the group with a reward above the others
    - with success chance $p$ per completion, a group of $G$ contains a hit with probability $1 - (1-p)^G$
    - for $p = 0.01$ and $G = 8$ that is about $8\%$
- a short SFT step on examples in the new style makes the format likely
    - SFT instills the format and patterns such as reflection and revision
    - it does not guarantee correct reasoning
- the DeepSeek-R1 paper also reports an experiment that applies RL straight to the base model, with no SFT (R1-Zero)
    - it learned to reason, but its outputs were hard to read and mixed languages


# The DeepSeek-R1 recipe, stage by stage

**Why this section?** One published recipe, seen as a sequence of the steps of RLLM-1.

> **Reasoning RL is the middle of a pipeline: SFT before it, more SFT and a second RL stage after it, and distillation at the end.**

<img src="images/rllm4_r1.png" width=85%>

- the stages, as the paper describes them
    - **cold-start SFT:** a small set of long reasoning examples gives a readable starting point
    - **reasoning RL:** GRPO with rule-based rewards (accuracy and format, and a language-consistency term)
    - **rejection sampling and SFT:** sample from the RL model, keep the correct completions, mix in non-reasoning data
    - **RL for helpfulness and harmlessness:** reward models for general prompts, rules for reasoning prompts
    - **distillation:** smaller models are fine-tuned on the large model's completions


**Why each stage**

- rejection sampling turns the RL model into a source of training data
    - it keeps only completions that pass the check
    - mixing in non-reasoning prompts avoids a model that only reasons
- the last RL stage adds the behaviors the reasoning stage ignored: helpful, honest, harmless
- distillation is SFT on the large model's outputs: a smaller model imitates it
    - it is imitation again (RLLM-1), now of a model rather than of a person


## SFT versus RL, again

**Why this section?** The same contrast as RLLM-1, now with a concrete case.

> **SFT shows the model one way to answer. A verifier accepts every way that works.**

- SFT imitates the demonstration
    - a different but correct completion counts as wrong for an exact-match target
    - it needs many demonstrations, usually written by people
- RL scores the model's own completions
    - many correct answers can earn the reward
    - the same prompt can be used again, and the reward can improve with each use
- caveats
    - the model learns whatever the reward encodes: a verifier checks only what its author wrote
    - demonstrations carry principles implicitly
- in practice: both, in order (SFT first)


## Outcome rewards and process rewards

**Why this section?** A reward for the whole completion is coarse. A reward for each step would give credit.

> **An outcome reward scores the whole completion; a process reward scores each step.**

- an **outcome reward** gives one reward at the end: right or wrong
    - it can come from a program or from a learned outcome reward model (ORM)
    - what the rule-based reward of this session does
- a **process reward model** (PRM) gives a reward for each reasoning step
    - more credit assignment, so more signal per completion
    - it needs a label for every step, which is costly, and a learned PRM can be hacked
- the DeepSeek-R1 paper reports not using a PRM for its reasoning RL
- GRPO's advantage is the same for every token: it uses an outcome reward
    - per-token credit is a theme of RLLM-5 and RLLM-6


# What we established, and what it sets up

- **Established**
    - a reasoning model thinks in a long chain of thought, with reflection and revision
    - the behavior is instilled in post-training
    - the reward is a program: accuracy and format checks, with no model to train
    - rewards are centered and scaled within the group (C2) to give advantages
    - a base model may never write the format, so reasoning RL starts from a short SFT step
    - the R1 recipe: cold-start SFT, reasoning RL, rejection sampling and SFT, RL for helpfulness and harmlessness, distillation
    - an outcome reward is coarse
    - a process reward gives credit per step but needs step labels
- **Sets up**
    - **the limitation:** the case study used plain GRPO, and real runs need more
    - next session, RLLM-5: PPO in depth
    - then RLLM-6: GRPO and its refinements
